In [3]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import numpy as np
import pandas as pd

import os
import time
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Input, BatchNormalization, Dropout

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader 
from torchvision import models, transforms

import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_DIR = "/kaggle/input/csiro-biomass"

In [4]:
# loading the dataset and looking at it
df = pd.read_csv(f"{IMG_DIR}/train.csv")
print("------------ train.csv ------------")
print(df.head)

print("------------ Unique Columns ------------")
print(df.columns)

print("------------ test.csv ------------")
# looking at the test data
df_test = pd.read_csv(f"{IMG_DIR}/test.csv")
print(df_test.head)

------------ train.csv ------------
<bound method NDFrame.head of                        sample_id              image_path Sampling_Date State  \
0     ID1011485656__Dry_Clover_g  train/ID1011485656.jpg      2015/9/4   Tas   
1       ID1011485656__Dry_Dead_g  train/ID1011485656.jpg      2015/9/4   Tas   
2      ID1011485656__Dry_Green_g  train/ID1011485656.jpg      2015/9/4   Tas   
3      ID1011485656__Dry_Total_g  train/ID1011485656.jpg      2015/9/4   Tas   
4            ID1011485656__GDM_g  train/ID1011485656.jpg      2015/9/4   Tas   
...                          ...                     ...           ...   ...   
1780   ID983582017__Dry_Clover_g   train/ID983582017.jpg      2015/9/1    WA   
1781     ID983582017__Dry_Dead_g   train/ID983582017.jpg      2015/9/1    WA   
1782    ID983582017__Dry_Green_g   train/ID983582017.jpg      2015/9/1    WA   
1783    ID983582017__Dry_Total_g   train/ID983582017.jpg      2015/9/1    WA   
1784          ID983582017__GDM_g   train/ID983582017.j

We are provided all these columns:
'sample_id', 'image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm', 'target_name', 'target'
And yet the test data doesn't provide these, it just provides one photo, if these columns were provided, then we would use an xgboost model

In [6]:
img = plt.imread(f"{IMG_DIR}/{df['image_path'][0]}")
#plt.imshow(img)
print(f"Image Size {img.shape}")
# check how many unique images
print(f'Number of unique images in train: {len(df["image_path"].unique())}')

Image Size (1000, 2000, 3)
Number of unique images in train: 357


'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm', does not need to exist in our dataframe because it is not provided in 'test.csv'.
The only thing provided in test.csv is the image path, so our goal here is. Given an image, we predict the five values.

In [7]:
# keep only these columns
df_slim = df[["image_path","target_name","target"]].copy()
print(f"df_slim.shape = {df_slim.shape}")
print(df_slim.head())

# pivot the table to make it look cleaner
wide = (
    df_slim.pivot_table(
        index="image_path",
        columns="target_name",
        values="target"
    ).reset_index()
)
wide.columns.name = None
print(f"wide.shape = {wide.shape}")
wide.head()

df_slim.shape = (1785, 3)
               image_path   target_name   target
0  train/ID1011485656.jpg  Dry_Clover_g   0.0000
1  train/ID1011485656.jpg    Dry_Dead_g  31.9984
2  train/ID1011485656.jpg   Dry_Green_g  16.2751
3  train/ID1011485656.jpg   Dry_Total_g  48.2735
4  train/ID1011485656.jpg         GDM_g  16.2750
wide.shape = (357, 6)


,image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
0,train/ID1011485656.jpg,0.0000,31.9984,16.2751,48.2735,16.2750
1,train/ID1012260530.jpg,0.0000,0.0000,7.6000,7.6000,7.6000
2,train/ID1025234388.jpg,6.0500,0.0000,0.0000,6.0500,6.0500
3,train/ID1028611175.jpg,0.0000,30.9703,24.2376,55.2079,24.2376
4,train/ID1035947949.jpg,0.4343,23.2239,10.5261,34.1844,10.9605


Now each row has the image path and the 5 values we want to predict

In [8]:
IMG_HEIGHT = 256
IMG_WIDTH = 512

# Get all X images
image_matrix_list = []
for image_path in wide['image_path']:
    with Image.open(f'{IMG_DIR}/{image_path}') as img:
        img = img.resize((IMG_WIDTH, IMG_HEIGHT))  # (width, height)
        img_array = np.array(img, dtype=np.float32) / 255.0  # normalize to [0,1]
        image_matrix_list.append(img_array)

# 357 images, 256x512, 3 color channels
X = np.array(image_matrix_list) # Shape (357, 256, 512, 3)
print(X.shape)

# Get all y labels
y = wide.iloc[:, 1:].values.astype(np.float32) # (All rows, ignore image path)
print("y.shape:", y.shape) # Shape (357, 5)

(357, 256, 512, 3)
y.shape: (357, 5)


In [9]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=5361)

In [13]:
# Implement a regular cnn
def create_model(height=IMG_HEIGHT, width=IMG_WIDTH, filters = [32]):
    model = tf.keras.models.Sequential()
    model.add(Input(shape=(height, width, 3)))

    for f in filters:
        model.add(Conv2D(f, (3,3), activation = "relu", padding='same'))
        model.add(BatchNormalization())
        model.add(MaxPooling2D((2,2)))
    
    model.add(Flatten())
    model.add(Dense(128, activation = "relu"))
    model.add(Dropout(.5))
    model.add(Dense(128, activation = "relu"))
    model.add(Dropout(.5))
    model.add(Dense(5, activation='linear')) # Linear output since this is regression
    return model

# Create model
model = create_model(filters = [16, 32, 64])
model.compile(optimizer='adam', loss='mse', metrics=['mse'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 256, 512, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256, 512, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 128, 256, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 128, 256, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 64, 128, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 128, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64, 128, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 32, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 131072)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │    16,777,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,818,533 (64.16 MB)

 Trainable params: 16,818,309 (64.16 MB)

 Non-trainable params: 224 (896.00 B)

In [14]:
# Start training CNN model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=16,
    verbose=1
)

Epoch 1/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 10s 228ms/step - loss: 3440.9233 - mse: 3440.9233 - val_loss: 1473.4077 - val_mse: 1473.4077
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - loss: 973.7944 - mse: 973.7944 - val_loss: 1158.0127 - val_mse: 1158.0127
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 812.9615 - mse: 812.9615 - val_loss: 971.9164 - val_mse: 971.9164
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - loss: 796.1425 - mse: 796.1425 - val_loss: 1177.8896 - val_mse: 1177.8896
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - loss: 700.2925 - mse: 700.2925 - val_loss: 1589.2042 - val_mse: 1589.2042
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 725.5500 - mse: 725.5500 - val_loss: 1803.6311 - val_mse: 1803.6311
Epoch 7/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 648.8909 - mse: 648.8909 - val_loss: 1975.8490 - val_mse: 1975.8490
Epoch 8/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 681.8917 - mse: 681.8917 - val_loss: 1787.7279 - v

In [16]:
pred = model.predict(X_test)

target_names = wide.columns[1:]
r2_col_score = []
for i, name in enumerate(target_names):
    r2 = r2_score(y_test[:, i], pred[:, i])
    print(f"{name}'s R2 score = {r2}")
    r2_col_score.append(r2)

mean_r2 = np.mean(r2_col_score)
print(f"Mean column r2 = {mean_r2}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Dry_Clover_g's R2 score = -0.24149602518899838
Dry_Dead_g's R2 score = -0.48554096168604066
Dry_Green_g's R2 score = -0.12895762274448552
Dry_Total_g's R2 score = -1.1156418316041887
GDM_g's R2 score = -0.5782098776113773
Mean column r2 = -0.5099692637670181


The baseline CNN performed bad on this dataset, likely due to the dataset being way too small for the cnn to learn any meaningful values.
As a result, the model was able to quickly overfit (loss is going down but the training loss being the same)
We will now switch to a prerained Vision Transformer which would hopefully improve the r2 score.

In [18]:
# Compute the sum of components
wide['calc_total'] = (
    wide['Dry_Clover_g'] +
    wide['Dry_Dead_g'] +
    wide['Dry_Green_g']
)

# check differnece between calculated and actual
wide['diff'] = wide['Dry_Total_g'] - wide['calc_total']

# show any rows with mismatches
mismatches = wide[wide['diff'].abs() > 1e-4]

print("Number of mismatches:", len(mismatches))
print(mismatches.head())

Number of mismatches: 23
                image_path  Dry_Clover_g  Dry_Dead_g  Dry_Green_g  \
27  train/ID1131079710.jpg       15.4794     11.1190       1.5261   
62  train/ID1343327476.jpg        3.1429      3.1429      37.7143   
69  train/ID1374789439.jpg        1.3556      1.3556      21.6889   
82  train/ID1428837636.jpg       10.1663     19.1366       3.8871   
92  train/ID1471216911.jpg       67.8977     11.4857      28.8667   

    Dry_Total_g    GDM_g  calc_total    diff  
27      28.1246  17.0055     28.1245  0.0001  
62      44.0000  40.8571     44.0001 -0.0001  
69      24.4000  23.0444     24.4001 -0.0001  
82      33.1901  14.0535     33.1900  0.0001  
92     108.2500  96.7643    108.2501 -0.0001  


Looking at this, we don't need to predict "Dry_Total_g" since that is just the sum of dry_clover_g + dry_dead_g + dry_green_g

In [29]:
target_cols_train = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'GDM_g'] # Model outputs
target_cols_eval = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']

train_df, val_df = train_test_split(wide, test_size=0.15, random_state=5361)

In [31]:
tfms = transforms.Compose([
    transforms.Resize((224,224)), # size that the vit accepts
    transforms.ToTensor(),
])

class BiomassDS(Dataset):
    def __init__(self, df, tfm):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{IMG_DIR}/{row['image_path']}").convert("RGB")
        img = self.tfm(img)
        y = torch.tensor(row[target_cols_train].values.astype("float32"))
        return img, y

train_ds = BiomassDS(train_df, tfms)
val_ds = BiomassDS(val_df, tfms)
# create dataloaders
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory = True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory = True)

In [38]:
class Model(nn.Module):
    def __init__(self, out_dim = 4):
        super().__init__()
        # load vit
        self.backbone = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        
        # replace classification head with regression head
        feat_dim = self.backbone.heads.head.in_features 
        self.backbone.heads.head = nn.Linear(feat_dim, out_dim) 
    
    def forward(self, img):
        return self.backbone(img)

model = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 5e-4)
loss_function = nn.MSELoss()

In [63]:
def train_vit(epochs = 50, verbose = 1):
    val_targets_full = torch.tensor(
        val_df[target_cols_eval].values.astype("float32"),
        device=device
    )  # shape: (N_val, 5)
    
    # start training
    for epoch in range(epochs):
        # ----- train -----
        model.train()
        for images, y_true in train_loader:
            images, y_true = images.to(device), y_true.to(device)
            optimizer.zero_grad()
            preds = model(images)
            loss = loss_function(preds, y_true)
            loss.backward()
            optimizer.step()
            
        # ----- eval -----
        model.eval()
        all_preds = []
        with torch.no_grad():
            for images, _ in val_loader:
                images = images.to(device)
                pred = model(images)
                all_preds.append(pred)
    
        all_preds = torch.cat(all_preds, dim=0) # (n,4)

        # ----- split predictions -----
        pred_clover = all_preds[:, 0]
        pred_dead   = all_preds[:, 1]
        pred_green  = all_preds[:, 2]
        pred_gdm    = all_preds[:, 3]

        # ----- calculate Dry_Total_g as sum of first 3 outputs -----
        pred_total = pred_clover + pred_dead + pred_green

        # ----- recreate matrix -----
        pred_full = torch.stack([pred_clover, pred_dead, pred_green, pred_total, pred_gdm], dim=1)

        y_true_np = val_targets_full.cpu().numpy()
        y_pred_np = pred_full.cpu().numpy()

        r2_values = r2_score(y_true_np, y_pred_np, multioutput='raw_values')
        r2_mean = r2_score(y_true_np, y_pred_np, multioutput='uniform_average')
        
        r2_dict = {t: float(r) for t, r in zip(target_cols_eval, r2_values)}

        if verbose == 1: # print every 5 epochs
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1} R2: {r2_dict} Mean: {r2_mean.mean():.4f}")

        elif verbose == 2: # print all epochs
            print(f"Epoch {epoch+1} R2: {r2_dict} Mean: {r2_mean.mean():.4f}")

In [ ]:
'''
Now, testing this vit model with:
no input normalization
no freezing layers
no added transforms like vertical flip or horizontal flips
'''

In [39]:
# since there is no freezing in any of the layers, this may take long time
start = time.time()
train_vit(epochs = 50, verbose = 2)
end = time.time()
print(f"Time taken: {end - start}")

Epoch 1 R2: {'Dry_Clover_g': -0.07290116776726085, 'Dry_Dead_g': -0.34817579415358657, 'Dry_Green_g': -0.5372537253056, 'Dry_Total_g': -1.5760789778461852, 'GDM_g': -1.1800271870341636} Mean: -0.7429
Epoch 2 R2: {'Dry_Clover_g': -0.022880409054219397, 'Dry_Dead_g': -0.11922427666872193, 'Dry_Green_g': -0.3212901693950685, 'Dry_Total_g': -0.7873103685104972, 'GDM_g': -0.7929233754555354} Mean: -0.4087
Epoch 3 R2: {'Dry_Clover_g': -0.017080837978544983, 'Dry_Dead_g': -0.007920025596964475, 'Dry_Green_g': -0.1751387259400814, 'Dry_Total_g': -0.35235850079238107, 'GDM_g': -0.49145478400898845} Mean: -0.2088
Epoch 4 R2: {'Dry_Clover_g': -0.002946564786944439, 'Dry_Dead_g': -0.004776660354005591, 'Dry_Green_g': -0.1128018410821825, 'Dry_Total_g': -0.19975177298981395, 'GDM_g': -0.27280685434400476} Mean: -0.1186
Epoch 5 R2: {'Dry_Clover_g': -0.03039671008242739, 'Dry_Dead_g': -0.005073089386109064, 'Dry_Green_g': -0.04504744440853803, 'Dry_Total_g': -0.14286306496952839, 'GDM_g': -0.18605769

In [ ]:
'''
Results:
Still not the best, but a lot better than the cnn defined earlier.
The mean r2 score is still negative.
CNN Mean: -0.5099692637670181
VIT Mean: -0.1678

Adding normalization in transforms and freeze some layers to make training a little faster
'''

In [84]:
# recreate the dataloaders but this time with normalization values as recommended by the documentation
train_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # --- NEW --- adding normalization here
])

train_ds = BiomassDS(train_df, train_tfms)
val_ds = BiomassDS(val_df, train_tfms)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory = True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory = True)

class Model(nn.Module):
    def __init__(self, out_dim = 4):
        super().__init__()
        # load vit
        self.backbone = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        
        # replace classification head with regression head
        feat_dim = self.backbone.heads.head.in_features 
        self.backbone.heads.head = nn.Linear(feat_dim, out_dim) 

        # --- NEW --- freeze all parameters
        for name, param in self.backbone.named_parameters():
            param.requires_grad = False

        # --- NEW --- unfreeze linear head
        for name, param in self.backbone.named_parameters():
            if 'heads.head' in name:
                param.requires_grad = True
        
        # --- NEW --- unfreeze last transformer block just for fun
        for name, param in self.backbone.named_parameters():
            if 'encoder.layers.11' in name:
                param.requires_grad = True
    
    def forward(self, img):
        return self.backbone(img)

# recreate the model
model = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 5e-4)
loss_function = nn.MSELoss()

In [85]:
# There is freezing weights this time + normalization
start = time.time()
train_vit(epochs = 50, verbose = 2)
end = time.time()
print(f"Time taken: {end - start}")

Epoch 1 R2: {'Dry_Clover_g': -0.20098882457343747, 'Dry_Dead_g': -0.86217880293904, 'Dry_Green_g': -1.0038464449451996, 'Dry_Total_g': -3.3763723465482505, 'GDM_g': -2.1886270247115625} Mean: -1.5264
Epoch 2 R2: {'Dry_Clover_g': -0.0854729482262766, 'Dry_Dead_g': -0.5429377389608094, 'Dry_Green_g': -0.8556588731598509, 'Dry_Total_g': -2.5907849440580417, 'GDM_g': -1.9359470680422017} Mean: -1.2022
Epoch 3 R2: {'Dry_Clover_g': -0.007670199651331311, 'Dry_Dead_g': -0.2960981627369468, 'Dry_Green_g': -0.7207728629416816, 'Dry_Total_g': -1.992459499282468, 'GDM_g': -1.7049493337643162} Mean: -0.9444
Epoch 4 R2: {'Dry_Clover_g': 0.048200954215072866, 'Dry_Dead_g': -0.10848475915899924, 'Dry_Green_g': -0.6058771565950896, 'Dry_Total_g': -1.5391070457029041, 'GDM_g': -1.501212237909161} Mean: -0.7413
Epoch 5 R2: {'Dry_Clover_g': 0.09197549321598186, 'Dry_Dead_g': 0.005783004130501146, 'Dry_Green_g': -0.5055399314960363, 'Dry_Total_g': -1.214017545018267, 'GDM_g': -1.317033291598816} Mean: -0.

In [73]:
# recreate the dataloaders with new transforms
train_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(), # --- NEW --- adding horizontal flip
    transforms.RandomVerticalFlip(),   # --- NEW --- adding vertical flip
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # --- NEW --- adding normalization here
])

val_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # --- NEW --- adding normalization here
])

train_ds = BiomassDS(train_df, train_tfms)
val_ds = BiomassDS(val_df, val_tfms)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory = True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory = True)

In [70]:
class Model(nn.Module):
    def __init__(self, out_dim = 4):
        super().__init__()
        # load vit
        self.backbone = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        
        # replace classification head with regression head
        feat_dim = self.backbone.heads.head.in_features 
        self.backbone.heads.head = nn.Linear(feat_dim, out_dim) 

        # --- NEW --- freeze all parameters
        for name, param in self.backbone.named_parameters():
            param.requires_grad = False

        # --- NEW --- unfreeze linear head
        for name, param in self.backbone.named_parameters():
            if 'heads.head' in name:
                param.requires_grad = True
        
        # --- NEW --- unfreeze last transformer block just for fun
        for name, param in self.backbone.named_parameters():
            if 'encoder.layers.11' in name:
                param.requires_grad = True
    
    def forward(self, img):
        return self.backbone(img)
# recreate the model
model = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 5e-4)
loss_function = nn.MSELoss()

In [45]:
# There is freezing weights this time + some extra transforms
start = time.time()
train_vit(epochs = 50, verbose = 2)
end = time.time()
print(f"Time taken: {end - start}")

Epoch 1 R2: {'Dry_Clover_g': -0.13623791838416355, 'Dry_Dead_g': -0.933687188950926, 'Dry_Green_g': -1.037785478547836, 'Dry_Total_g': -3.385498291104203, 'GDM_g': -2.180140427776285} Mean: -1.5347
Epoch 2 R2: {'Dry_Clover_g': -0.04109611850543993, 'Dry_Dead_g': -0.5837193250860229, 'Dry_Green_g': -0.8867994159766037, 'Dry_Total_g': -2.613019747970926, 'GDM_g': -1.9302221540882738} Mean: -1.2110
Epoch 3 R2: {'Dry_Clover_g': 0.02100807345452682, 'Dry_Dead_g': -0.3307318521372251, 'Dry_Green_g': -0.7535290378231807, 'Dry_Total_g': -2.0423823833701955, 'GDM_g': -1.7060433145242073} Mean: -0.9623
Epoch 4 R2: {'Dry_Clover_g': 0.07215216090233845, 'Dry_Dead_g': -0.1502280511108176, 'Dry_Green_g': -0.635997685277347, 'Dry_Total_g': -1.6019893470968198, 'GDM_g': -1.5023329000265502} Mean: -0.7637
Epoch 5 R2: {'Dry_Clover_g': 0.11038120624582726, 'Dry_Dead_g': -0.021680831806186962, 'Dry_Green_g': -0.5334472320742492, 'Dry_Total_g': -1.273151032071954, 'GDM_g': -1.3204230949361175} Mean: -0.607

In [ ]:
'''
Without normalization or freezing, the ViT improves gradually but r2 score remains negative overall, reaching a mean of –0.1678.
With proper normalization and freezing of most layers, the model trains much better and gets a positive mean r2 score, reaching around 0.2758 by epoch 30. 
If I continue to run this with more epochs, I would assume that the mean r2 score would climb more.
'''

In [46]:
# Running the vit another 20 times, so technically epoch 51-70
start = time.time()
train_vit(20, verbose=1)
end = time.time()
print(f"Time taken: {end - start}")
# Final mean at epoch 70 = 0.3052

Epoch 5 R2: {'Dry_Clover_g': 0.553758358868742, 'Dry_Dead_g': 0.48185348514306814, 'Dry_Green_g': 0.20931537073311257, 'Dry_Total_g': 0.13506585896491452, 'GDM_g': 0.029912576477754693} Mean: 0.2820
Epoch 10 R2: {'Dry_Clover_g': 0.5658306453461323, 'Dry_Dead_g': 0.4899543402824128, 'Dry_Green_g': 0.22100911079696972, 'Dry_Total_g': 0.1539540709813777, 'GDM_g': 0.039997225532095615} Mean: 0.2941
Epoch 15 R2: {'Dry_Clover_g': 0.573901298025868, 'Dry_Dead_g': 0.4936349690415561, 'Dry_Green_g': 0.22916760578527073, 'Dry_Total_g': 0.15992767894842996, 'GDM_g': 0.04621283332809056} Mean: 0.3006
Epoch 20 R2: {'Dry_Clover_g': 0.5805379611177925, 'Dry_Dead_g': 0.4984751851532988, 'Dry_Green_g': 0.2371340012734, 'Dry_Total_g': 0.15977638881258438, 'GDM_g': 0.05008763908490044} Mean: 0.3052
Time taken: 274.65475702285767


In [ ]:
'''
# possible data augmentations improvements
Give it a random rotation
Give it a random resize crop
'''

In [57]:
# testing data augmentations mentioned above on our transforms
train_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # --- NEW ---
    transforms.RandomRotation(degrees=10),               # --- NEW ---
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# recreate dataloaders
train_ds = BiomassDS(train_df, train_tfms)
val_ds = BiomassDS(val_df, val_tfms)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory = True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory = True)

In [56]:
# define a new model to test augmentations
model = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 5e-4)
loss_function = nn.MSELoss()

In [58]:
start = time.time()
train_vit(epochs = 50, verbose=2)
end = time.time()
print(f"Time taken: {end - start}")

Epoch 1 R2: {'Dry_Clover_g': -0.15131414762244444, 'Dry_Dead_g': -0.8113656025520193, 'Dry_Green_g': -1.041917468708074, 'Dry_Total_g': -3.3257421643223903, 'GDM_g': -2.2258168492308092} Mean: -1.5112
Epoch 2 R2: {'Dry_Clover_g': -0.04976998228193308, 'Dry_Dead_g': -0.4815440644142268, 'Dry_Green_g': -0.8947960433912152, 'Dry_Total_g': -2.555667210918063, 'GDM_g': -1.9758967778551977} Mean: -1.1915
Epoch 3 R2: {'Dry_Clover_g': 0.019296584847057985, 'Dry_Dead_g': -0.24801365258169739, 'Dry_Green_g': -0.7587186400680563, 'Dry_Total_g': -1.965460998198075, 'GDM_g': -1.7426380415687097} Mean: -0.9391
Epoch 4 R2: {'Dry_Clover_g': 0.06821335788806793, 'Dry_Dead_g': -0.08092229359434411, 'Dry_Green_g': -0.6413816348914305, 'Dry_Total_g': -1.5397801557077155, 'GDM_g': -1.5369927496739586} Mean: -0.7462
Epoch 5 R2: {'Dry_Clover_g': 0.10576354071976457, 'Dry_Dead_g': 0.027988589872632863, 'Dry_Green_g': -0.5438039341909011, 'Dry_Total_g': -1.224066246023194, 'GDM_g': -1.3564189254567594} Mean: -

In [ ]:
'''
Added RandomResizedCrop + RandomRotation made the model slightly worse
EPOCH = 50
Model with RandomResizedCrop + RandomRotation: 0.2456
Model without the above mentioned transforms: 0.2758
'''

In [78]:
# testing data augmentations without vertical flip
train_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# recreate dataloaders
train_ds = BiomassDS(train_df, train_tfms)
val_ds = BiomassDS(val_df, val_tfms)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory = True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory = True)

# define a new model to test augmentations
model = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 5e-4)
loss_function = nn.MSELoss()

In [80]:
start = time.time()
train_vit(epochs = 50, verbose=2)
end = time.time()
print(f"Time taken: {end - start}")

Epoch 1 R2: {'Dry_Clover_g': -0.1401019007171862, 'Dry_Dead_g': -0.829873032026875, 'Dry_Green_g': -1.0788748004427209, 'Dry_Total_g': -3.4239662181843276, 'GDM_g': -2.217263519180346} Mean: -1.5380
Epoch 2 R2: {'Dry_Clover_g': -0.04003782129340272, 'Dry_Dead_g': -0.4988083172759388, 'Dry_Green_g': -0.9224433887951162, 'Dry_Total_g': -2.637719313613748, 'GDM_g': -1.9624445300896185} Mean: -1.2123
Epoch 3 R2: {'Dry_Clover_g': 0.03204512289070294, 'Dry_Dead_g': -0.26503356510857445, 'Dry_Green_g': -0.7822374488818902, 'Dry_Total_g': -2.042685543783632, 'GDM_g': -1.7295185562929758} Mean: -0.9575
Epoch 4 R2: {'Dry_Clover_g': 0.08247220049724913, 'Dry_Dead_g': -0.0964107095501181, 'Dry_Green_g': -0.6633594799186624, 'Dry_Total_g': -1.6157063514975132, 'GDM_g': -1.527777648918347} Mean: -0.7642
Epoch 5 R2: {'Dry_Clover_g': 0.12439932801804243, 'Dry_Dead_g': 0.02169245572942491, 'Dry_Green_g': -0.5566727709053216, 'Dry_Total_g': -1.2829183431014433, 'GDM_g': -1.342594945864498} Mean: -0.6072

In [81]:
# train another 50 times
start = time.time()
train_vit(epochs = 50, verbose=1)
end = time.time()
print(f"Time taken: {end - start}")

Epoch 5 R2: {'Dry_Clover_g': 0.5630799528269737, 'Dry_Dead_g': 0.49275423621536274, 'Dry_Green_g': 0.21629907487381173, 'Dry_Total_g': 0.1422083414943366, 'GDM_g': 0.01967342472397582} Mean: 0.2868
Epoch 10 R2: {'Dry_Clover_g': 0.571565643859944, 'Dry_Dead_g': 0.4976013168074639, 'Dry_Green_g': 0.22569676548375406, 'Dry_Total_g': 0.14788211986130118, 'GDM_g': 0.024303139890981762} Mean: 0.2934
Epoch 15 R2: {'Dry_Clover_g': 0.5823132705386536, 'Dry_Dead_g': 0.5022446859302405, 'Dry_Green_g': 0.23527913539795853, 'Dry_Total_g': 0.15745488199559343, 'GDM_g': 0.03189935352080442} Mean: 0.3018
Epoch 20 R2: {'Dry_Clover_g': 0.5875729301612188, 'Dry_Dead_g': 0.504746941667547, 'Dry_Green_g': 0.2428199728690259, 'Dry_Total_g': 0.1582143964598668, 'GDM_g': 0.03555002304047872} Mean: 0.3058
Epoch 25 R2: {'Dry_Clover_g': 0.5974212524332188, 'Dry_Dead_g': 0.5078544959947513, 'Dry_Green_g': 0.25020415409287566, 'Dry_Total_g': 0.1664611515707053, 'GDM_g': 0.04280114958211445} Mean: 0.3129
Epoch 30 R

In [ ]:
'''
Not having vertical flip does a tiny bit better, but it probably is just due to randomness.
vertical flip + horizontal flip = 0.2758
horizontal flip only = 0.2767

at epoch 100, mean r2 score = 0.3393

Since torchvision viT forces images to be 224x224, we will use a different model accepts different shapes.
'''

In [82]:
# changing resize shape
train_tfms = transforms.Compose([
    transforms.Resize((384,768)), # from 224x224 -> 384x768 (originally 1000x2000)
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # --- NEW --- adding normalization here
])

val_tfms = transforms.Compose([
    transforms.Resize((384,768)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # --- NEW --- adding normalization here
])

train_ds = BiomassDS(train_df, train_tfms)
val_ds = BiomassDS(val_df, val_tfms)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory = True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory = True)


class Model(nn.Module):
    def __init__(self, out_dim = 4):
        super().__init__()
        # load vit
        self.backbone = timm.create_model('vit_base_patch16_384', pretrained=True, num_classes=out_dim, img_size=(384, 768))
        self.backbone.patch_embed.strict_img_size = False # not square image

        # --- NEW --- freeze all parameters
        for name, param in self.backbone.named_parameters():
            param.requires_grad = False

        # --- NEW --- unfreeze linear head
        for name, param in self.backbone.named_parameters():
            if name.startswith('head'):
                param.requires_grad = True
        
        # --- NEW --- unfreeze last transformer block just for fun
        for name, param in self.backbone.named_parameters():
            if name.startswith('blocks.11'):
                param.requires_grad = True
    
    def forward(self, img):
        return self.backbone(img)

# create new model that uses bigger size photos
model = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 5e-4)
loss_function = nn.MSELoss()

In [83]:
start = time.time()
train_vit(epochs=50, verbose = 2)
end = time.time()
print(f"Total time: {end - start}")

Epoch 1 R2: {'Dry_Clover_g': 0.5696522898431884, 'Dry_Dead_g': 0.2708139429760591, 'Dry_Green_g': 0.2724415035821405, 'Dry_Total_g': -0.03646397575873883, 'GDM_g': -0.0553877387622137} Mean: 0.2042
Epoch 2 R2: {'Dry_Clover_g': 0.6128397697433412, 'Dry_Dead_g': 0.3322855859660335, 'Dry_Green_g': 0.3915146367518314, 'Dry_Total_g': 0.10794308616953285, 'GDM_g': 0.11657158872131756} Mean: 0.3122
Epoch 3 R2: {'Dry_Clover_g': 0.5695729875424536, 'Dry_Dead_g': 0.3934444930106975, 'Dry_Green_g': 0.3572460065486832, 'Dry_Total_g': 0.18490691860809116, 'GDM_g': 0.17983009467741462} Mean: 0.3370
Epoch 4 R2: {'Dry_Clover_g': 0.6604657145454457, 'Dry_Dead_g': 0.4151217730611727, 'Dry_Green_g': 0.5118528363841615, 'Dry_Total_g': 0.2360407409643216, 'GDM_g': 0.2863868743844057} Mean: 0.4220
Epoch 5 R2: {'Dry_Clover_g': 0.6365109227265979, 'Dry_Dead_g': 0.4149290492530635, 'Dry_Green_g': 0.5546992555296535, 'Dry_Total_g': 0.4384107882273479, 'GDM_g': 0.40496032634735735} Mean: 0.4899
Epoch 6 R2: {'Dry

In [ ]:
'''
This model seemed to have converged really quick. Mean r2 score peaked at epoch = 6, after that, I assume it started overfitting.
The total time to train this takes a long time though.
'''

In [86]:
# changing resize shape
train_tfms = transforms.Compose([
    transforms.Resize((768,1536)), # from 224x224 -> 768x1536 (originally 1000x2000)
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((768,1536)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_ds = BiomassDS(train_df, train_tfms)
val_ds = BiomassDS(val_df, val_tfms)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=4, pin_memory = True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=4, pin_memory = True)


class Model(nn.Module):
    def __init__(self, out_dim = 4):
        super().__init__()
        # load vit
        self.backbone = timm.create_model('vit_base_patch16_384', pretrained=True, num_classes=out_dim, img_size=(768,1536))
        self.backbone.patch_embed.strict_img_size = False # not square image

        # --- NEW --- freeze all parameters
        for name, param in self.backbone.named_parameters():
            param.requires_grad = False

        # --- NEW --- unfreeze linear head
        for name, param in self.backbone.named_parameters():
            if name.startswith('head'):
                param.requires_grad = True
        
        # --- NEW --- unfreeze last transformer block just for fun
        for name, param in self.backbone.named_parameters():
            if name.startswith('blocks.11'):
                param.requires_grad = True
    
    def forward(self, img):
        return self.backbone(img)

# create new model that uses bigger size photos
model = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = 5e-4)
loss_function = nn.MSELoss()

In [87]:
start = time.time()
train_vit(epochs=5, verbose = 2)
end = time.time()
print(f"Total time: {end - start}")

Epoch 1 R2: {'Dry_Clover_g': 0.4673388223471455, 'Dry_Dead_g': 0.29911507824499217, 'Dry_Green_g': 0.12679875264948626, 'Dry_Total_g': -0.23941644235642623, 'GDM_g': -0.10703007551572363} Mean: 0.1094
Epoch 2 R2: {'Dry_Clover_g': 0.4988290654901626, 'Dry_Dead_g': 0.35538931441535626, 'Dry_Green_g': 0.20601531445553078, 'Dry_Total_g': 0.0506345218186125, 'GDM_g': -0.03860868796364625} Mean: 0.2145
Epoch 3 R2: {'Dry_Clover_g': 0.6962221648284724, 'Dry_Dead_g': 0.3003923358755155, 'Dry_Green_g': 0.45159420228510494, 'Dry_Total_g': 0.24546510313056802, 'GDM_g': 0.24003103910890444} Mean: 0.3867
Epoch 4 R2: {'Dry_Clover_g': 0.7294252986335563, 'Dry_Dead_g': 0.38931828200617213, 'Dry_Green_g': 0.5075805241872251, 'Dry_Total_g': 0.4280648492217767, 'GDM_g': 0.3108000974466586} Mean: 0.4730
Epoch 5 R2: {'Dry_Clover_g': 0.7243605553653676, 'Dry_Dead_g': 0.46842272359719483, 'Dry_Green_g': 0.5438390896980894, 'Dry_Total_g': 0.40267954547545337, 'GDM_g': 0.32887843349138} Mean: 0.4936
Total time:

In [ ]:
'''----------------- SUMMARY -----------------
224x224 at patch 16 outputs 196 patches
384x768 at patch 16 outputs 1152 patches
768x1536 at patch 16 outputs 4608 patches

AT EPOCH = 50
(A) Baseline CNN
Mean: = -0.5099692637670181

(B) ViT without normalization or freezing weights:
Mean: -0.1678

(C) ViT with normalization and freezing weights:
Mean: 0.2750

(D) ViT + norm + freeze + vertical and horizontal flip:
Mean: 0.2758

(E) ViT + norm + freeze + horizontal flip
Mean: 0.2767

(F) Same ViT as (D) and with RandomResizedCrop + RandomRotation
Mean: 0.2456

(G) Different ViT with greater image size (from 224x224 -> 384x768)
Mean: 0.4950

(H) Greater image size again (from 384x768 -> 768x1536)
Mean: 0.4936 (ONLY 5 EPOCHS)
'''

In [ ]:
'''
REFERENCES & PLACES WHERE CODE WAS USED

https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html
https://docs.pytorch.org/vision/stable/models.html
https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vit_b_16.html#torchvision.models.ViT_B_16_Weights
https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vit_b_16.html
https://docs.pytorch.org/vision/stable/transforms.html
https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html
https://docs.pytorch.org/vision/main/generated/torchvision.transforms.RandomResizedCrop.html
https://timm.fast.ai/models
'''